# 06 — Resolve semantic rules with an LLM judge

**Foundry feature:** the **LLM-judge evaluation** layer behind Foundry's eval-model scoring, plus
the self-preference-bias check the wizard's model pickers don't do for you automatically.
**Mode: CLI.**

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## Why this step exists

Regex-backed rules (`regex`, `all_of_regex`, `any_of_regex`, `absent`) are checked automatically in
notebook 05. `semantic` rules — "does this convey the right idea," not "does this exact string
appear" — are never silently passed. They sit in an `UNJUDGED` queue until a judge (or a human)
resolves them. That's deliberate: for several of this pack's other agents, most of the interesting
gaps are semantic, and a validator that silently passed them would report even a bad baseline as
mostly promotable.

## Free smoke test — the stub judge

`--judge-backend stub` uses a zero-cost, zero-network lexical-overlap heuristic. It proves the
resolution *plumbing* runs end-to-end; it has no real understanding, and its verdicts are never
evidence about a candidate's quality — only ever use it to check wiring before spending money on a
real judge.

In [ ]:
candidate_path = AGENT_DIR / "candidates" / "foundry_run1.md"

run(["python3", "_tools/validate_candidate.py",
     "--agent", AGENT_ID,
     "--candidate", str(candidate_path),
     "--candidate-source", "optimize",
     "--judge-backend", "stub"])

Every `[??]` from notebook 05 should now read `[OK]` or `[XX]` with a `[judged by stub-judge ...]`
tag and a list of per-repeat votes (`judge_config.repeats_per_item` majority-voting).

## Real judge resolution (optional, costs money)

Using the agent's own `judge_config` — `primary_judge_model` pinned to a vendor family disjoint from
every supported Foundry optimization model, plus `cross_judge_model`, deliberately same-family as
one optimization condition, run in parallel to *measure* — not correct for — self-preference bias.

```bash
pip install litellm   # only needed for --judge-backend litellm
export ANTHROPIC_API_KEY=...   # or whichever provider judge_config.primary_judge_model needs

python _tools/validate_candidate.py \
  --agent 01-travel-approval-strict \
  --candidate 01-travel-approval-strict/candidates/foundry_run1.md \
  --candidate-source optimize \
  --judge-backend litellm --cross-judge \
  --json 01-travel-approval-strict/candidates/foundry_run1.judged.json
```

Read `primary_cross_judge_agreement` (a corpus-level Cohen's kappa) in the resulting JSON before
trusting the primary judge's verdicts on this run — a low kappa means the two judge families
disagree enough that you shouldn't treat either one's score as ground truth on its own.

In [ ]:
# Uncomment once ANTHROPIC_API_KEY (or your provider's key) is set and `pip install litellm` has run:

# run(["python3", "_tools/validate_candidate.py",
#      "--agent", AGENT_ID,
#      "--candidate", str(candidate_path),
#      "--candidate-source", "optimize",
#      "--judge-backend", "litellm", "--cross-judge",
#      "--json", str(candidate_path.with_suffix(".judged.json"))])
print("Real judge run left commented out -- see the markdown cell above for prerequisites.")

## Promotion gate, so far

A candidate is promotable only once `blocked` is `false` **and** every `UNJUDGED` item has been
resolved to a non-critical-failure verdict — see
`../prompt-agent-optimizer-baselines/docs/agent-evaluation-guide.md#what-counts-as-a-promotable-candidate`
for the full checklist. That still isn't the whole story: everything so far has scored the candidate
against `dataset/optimize.jsonl`, the split the optimizer already saw. Continue to
**`07_evaluate_optimized_candidate.ipynb`** for the held-out half.